### Задача 1

Анализ рынка вакансий

Предоставлен файл vacs.json, который содержит информацию о вакансиях, в частности, данные о предлагаемых зарплатах, даты размещения, описание

Предлагается ознакомиться с данными и найти:
- 10 самых новых вакансий
- 10 вакансий с максимальной годовой заработной платой
- среднюю зарплату
- все возможные типы занятости (полная занятость / совместительство)
- число вакансий, подходящих для старта (карьеры) и среднюю зарплату по ним
- число вакансий, требующих максимального опыта и среднюю зарплату по ним

Это задание лучше выполнять в Юпитер-ноутбуке построчно

In [29]:
import json
from datetime import datetime

with open('vacs.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

def get_date(vac):
    date = vac['data']['add_date']
    return datetime.fromisoformat(date)

sorted_for_add_date = sorted(data, key=get_date, reverse=True)
for i in sorted_for_add_date[:10]:
    print(i['data']['add_date'])


def get_max_sal(vac):
    max_sal = vac['data']['salary_max_rub']
    if max_sal is not None:
        return max_sal

    min_sal = vac['data']['salary_min_rub']
    if min_sal is not None:
        return min_sal
    
    return 0
    

sorted_for_sal = sorted(data, key=get_max_sal, reverse=True)
for i in sorted_for_sal[:10]:
    print(i['data']['salary_max_rub'] if i['data']['salary_max_rub'] is not None else i['data']['salary_min_rub'])


def get_avg_sal(vac):
    min_rub = vac['data']['salary_min_rub']
    max_rub = vac['data']['salary_max_rub']

    if min_rub is not None and max_rub is not None:
        return (min_rub + max_rub) / 2
    elif min_rub is not None:
        return min_rub
    elif max_rub is not None:
        return max_rub
    else:
        return None
    
salaries = []
for vac in data:
    s = get_avg_sal(vac)
    if s is not None:
        salaries.append(s)

print(f"средняя зп {sum(salaries) / len(salaries) / 12}")


def get_working_type(vac):
    return vac['data']['working_type']['title']

work_types = []
for i in data:
    if get_working_type(i) not in work_types:
        work_types.append(get_working_type(i))

print(f"Виды занятости: {work_types}")


vac_for_new = []
sum_rub = 0.0

for vac in data:
    if vac['data']['experience_length']['title'] == 'без опыта':
        if get_avg_sal(vac) is not None:
            vac_for_new.append(vac)
            sum_rub += get_avg_sal(vac)

        

print(f"Вакансий для новичков - {len(vac_for_new)}, средняя зарплата - {sum_rub/len(vac_for_new)}")


def get_exp_id(vac):
    return vac['data']['experience_length']['id']

sorted_exp = sorted(data, key=get_exp_id, reverse=True)
max_exp = sorted_exp[0]['data']['experience_length']['id']


max_exp_sal = 0
max_exp_cout = 0

for i in sorted_exp:
    if get_exp_id(i) != max_exp:
        break

    if get_avg_sal(i) is not None:
        max_exp_sal += get_avg_sal(i)
        max_exp_cout += 1

print(f"Наибольший опыт - {sorted_exp[0]['data']['experience_length']['title']}, число вакансий - {max_exp_cout}, средняя зп - {max_exp_sal/max_exp_cout}")

2023-01-20T16:55:22+03:00
2023-01-20T16:51:27+03:00
2023-01-20T16:48:14+03:00
2023-01-20T16:43:56+03:00
2023-01-20T16:43:20+03:00
2023-01-20T16:42:09+03:00
2023-01-20T16:36:41+03:00
2023-01-20T16:36:41+03:00
2023-01-20T16:36:41+03:00
2023-01-20T16:36:41+03:00
7700000
5600000
5600000
5600000
5600000
3500000
3500000
3500000
3500000
3304000
средняя зп 84527.75541227145
Виды занятости: ['полная занятость', 'частичная занятость', 'работа вахтовым методом', 'временная работа / freelance', 'стажировка']
Вакансий для новичков - 554, средняя зарплата - 981763.9747292419
Наибольший опыт - более 6 лет, число вакансий - 14, средняя зп - 1858500.0


### Задача 2

Подсистема аутентификации

Требуется написать класс, который будет отвечать за аутентификацию пользователя в некоторой системе.

Класс должен содержать методы:
- для аутентификации пользователя по логину и паролю, возвращаемое значение типа bool (авторизован / нет)
- для добавления нового пользователя (пары логин-пароль)
- для смены пароля

Данные должны храниться перманентно, то есть не в памяти, а в каком-либо хранилище... для простоты можно использовать файл

Пароль должен быть защищён



In [5]:
import pickle
import hashlib
import os
from typing import Dict

class AuthenticationSystem:
    def __init__(self):
        self.storage_file = "users.pkl"
        self.users: Dict[str, str] = {}
        self._load_data()

    def _hash_pass(self, password: str) -> str:
        return hashlib.sha256(password.encode('utf-8')).hexdigest()
    
    def _load_data(self) -> None:
        if os.path.exists(self.storage_file):
            with open(self.storage_file, 'rb') as f:
                self.users = pickle.load(f)

    def _save_data(self) -> None:
        with open(self.storage_file, 'wb') as f:
            pickle.dump(self.users, f)

    def authenticate(self, login: str, password: str) -> bool:
        return login in self.users and self.users[login] == self._hash_pass(password)
    
    def add_user(self, login: str, password: str) -> bool:
        if login in self.users:
            return False
        self.users[login] = self._hash_pass(password)
        self._save_data()
        return True
    
    def change_password(self, login: str, old_password: str, new_password: str) -> bool:
        if self.authenticate(login, old_password):
            self.users[login] = self._hash_pass(new_password)
            self._save_data()
            return True
        return False
    
    def lists_users(self) -> list:
        return list(self.users.keys())

if __name__ == "__main__":
    auth = AuthenticationSystem()
    auth.add_user('user1', 'pass123')
    auth.add_user('user2', 'secret456')
    auth.add_user('user1', 'newpass')

    print(f"cписок пользователей: {auth.lists_users()}\n")


    print(f"с правильным паролем: {auth.authenticate('user1', 'pass123')}")
    print(f"с неправильным паролем: {auth.authenticate('user1', 'wrong')}")

cписок пользователей: ['user1', 'user2']

с правильным паролем: True
с неправильным паролем: False
